# LSTM & GRU

**Companion lesson:** https://ml-viz.vercel.app/courses/rnns/03-lstm-and-gru

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## An LSTM cell, fully from scratch

Three gates and an **additive** cell-state update — the gradient highway that fixes the vanishing gradient from the previous lesson.

In [ ]:
def sigmoid(z): return 1/(1+np.exp(-z))

class LSTMCell:
    def __init__(self, n_in, nh):
        k = n_in + nh
        self.nh = nh
        self.Wf=np.random.randn(nh,k)*0.1; self.bf=np.ones((nh,1))   # forget bias=1
        self.Wi=np.random.randn(nh,k)*0.1; self.bi=np.zeros((nh,1))
        self.Wc=np.random.randn(nh,k)*0.1; self.bc=np.zeros((nh,1))
        self.Wo=np.random.randn(nh,k)*0.1; self.bo=np.zeros((nh,1))

    def step(self, x, h, c):
        z = np.vstack([h, x])
        f = sigmoid(self.Wf@z + self.bf)
        i = sigmoid(self.Wi@z + self.bi)
        g = np.tanh(self.Wc@z + self.bc)
        o = sigmoid(self.Wo@z + self.bo)
        c = f*c + i*g                 # additive update = gradient highway
        h = o*np.tanh(c)
        return h, c, dict(f=f, i=i, o=o)

cell = LSTMCell(n_in=3, nh=5)
h = c = np.zeros((5,1))
for t in range(4):
    h, c, gates = cell.step(np.random.randn(3,1), h, c)
    print(f't={t}  mean forget gate={gates["f"].mean():.2f}  ||cell||={np.linalg.norm(c):.2f}')

## A full LSTM timestep, by hand (pure-Python verification)

The lesson works one scalar LSTM step ($D=1$, $H=1$) entirely by hand. Here we reproduce it with only the standard library (`math`), so the arithmetic is deterministic and checkable without any dependencies.

Inputs: $h_{t-1}=0$, $x_t=1$, $c_{t-1}=2$. Weights are chosen to keep the pre-activations simple.

In [ ]:
import math  # stdlib only -- deterministic, no numpy needed

def sig(z):
    return 1 / (1 + math.exp(-z))

# Inputs
h_prev, x, c_prev = 0.0, 1.0, 2.0

# Weights per gate: (W_h, W_x, bias)
Wf = (0.0, 0.0,  2.0)   # forget  (bias +2 -> defaults to remembering)
Wi = (0.0, 1.0, -1.0)   # input
Wc = (0.0, 1.0,  0.0)   # candidate
Wo = (0.0, 1.0,  0.0)   # output

def preact(W):
    Wh, Wx, b = W
    return Wh * h_prev + Wx * x + b

f  = sig(preact(Wf))                 # forget gate
i  = sig(preact(Wi))                 # input gate
ct = math.tanh(preact(Wc))           # candidate
o  = sig(preact(Wo))                 # output gate

c = f * c_prev + i * ct              # additive cell update
h = o * math.tanh(c)                 # hidden state

print(f"f_t      = {f:.3f}   (expected 0.881)")
print(f"i_t      = {i:.3f}   (expected 0.500)")
print(f"c_tilde  = {ct:.3f}   (expected 0.762)")
print(f"o_t      = {o:.3f}   (expected 0.731)")
print(f"c_t      = {c:.3f}   (expected 2.142)")
print(f"h_t      = {h:.3f}   (expected 0.711)")

# Verification against the hand-computed values in the lesson
assert abs(f  - 0.881) < 1e-3
assert abs(i  - 0.500) < 1e-3
assert abs(ct - 0.762) < 1e-3
assert abs(o  - 0.731) < 1e-3
assert abs(c  - 2.142) < 1e-3
assert abs(h  - 0.711) < 1e-3
print("\nVERIFIED: by-hand LSTM timestep matches the lesson.")

## Parameter count: LSTM vs GRU

Each gate is a linear layer over the concatenation $[\mathbf{h}_{t-1}, \mathbf{x}_t]$, costing $H(D+H+1)$ parameters ($W_x$: $H\times D$, $W_h$: $H\times H$, bias: $H$). An LSTM has **4** such layers (forget, input, candidate, output); a GRU has **3** (reset, update, candidate). Pure-Python count below.

In [ ]:
def gate_params(D, H):
    """Parameters in one gate layer: W_x (H*D) + W_h (H*H) + bias (H)."""
    return H * D + H * H + H

def lstm_params(D, H):
    return 4 * gate_params(D, H)

def gru_params(D, H):
    return 3 * gate_params(D, H)

D, H = 10, 20
per_gate = gate_params(D, H)
lstm_p = lstm_params(D, H)
gru_p = gru_params(D, H)

print(f"D={D}, H={H}")
print(f"per-gate layer : {per_gate}        (= {H}*{D} + {H}*{H} + {H})")
print(f"LSTM (4 gates) : {lstm_p}")
print(f"GRU  (3 gates) : {gru_p}")
print(f"GRU / LSTM     : {gru_p / lstm_p:.2f}   ({(1 - gru_p/lstm_p)*100:.0f}% fewer)")

assert per_gate == 620
assert lstm_p == 2480
assert gru_p == 1860
assert abs(gru_p / lstm_p - 0.75) < 1e-9
print("\nVERIFIED: parameter counts match the lesson (2480 vs 1860, exactly 25% fewer).")

## The gradient highway, quantified (pure-Python)

Differentiating the additive update gives $\partial c_t / \partial c_{t-1} \approx f_t$, so the gradient over $T$ steps is the **product of forget gates**, $\prod_t f_t \approx f^T$. A vanilla RNN instead scales like (per-step factor)$^T$. The numbers below match the contrast in the lesson.

In [ ]:
T = 50

# LSTM cell-state gradient over T steps = product of forget gates (held constant here)
lstm_open   = 0.95 ** T   # forget gate kept open at 0.95
lstm_wide   = 0.99 ** T   # forget gate kept at 0.99
# Vanilla RNN: gradient scales like (effective recurrent factor)^T
rnn_decay   = 0.90 ** T

print(f"After T={T} steps:")
print(f"  LSTM, forget gate f=0.95 : {lstm_open:.4f}")
print(f"  LSTM, forget gate f=0.99 : {lstm_wide:.4f}")
print(f"  vanilla RNN, factor 0.90 : {rnn_decay:.6f}")
print(f"  LSTM(0.95) is ~{lstm_open / rnn_decay:.0f}x larger than the RNN gradient")

assert abs(lstm_open - 0.0769) < 1e-3
assert abs(lstm_wide - 0.6050) < 1e-3
assert abs(rnn_decay - 0.005154) < 1e-5
assert lstm_open / rnn_decay > 10
print("\nVERIFIED: gradient-highway numbers match the lesson (15x advantage at f=0.95).")

## A GRU cell, from scratch

Two gates (reset, update), one state vector — the update gate does the forget+input job.

In [ ]:
class GRUCell:
    def __init__(self, n_in, nh):
        k = n_in + nh; self.nh = nh
        self.Wz=np.random.randn(nh,k)*0.1; self.Wr=np.random.randn(nh,k)*0.1
        self.Wh=np.random.randn(nh,k)*0.1

    def step(self, x, h):
        z = sigmoid(self.Wz@np.vstack([h,x]))
        r = sigmoid(self.Wr@np.vstack([h,x]))
        hh = np.tanh(self.Wh@np.vstack([r*h, x]))
        return (1-z)*h + z*hh

gru = GRUCell(3, 5); h = np.zeros((5,1))
for t in range(4):
    h = gru.step(np.random.randn(3,1), h)
print('GRU final hidden:', np.round(h.ravel(), 3))

## The whole point: memory survives long gaps

Compare how a signal injected at $t=0$ persists in an LSTM cell state vs a vanilla RNN hidden state, when the forget gate stays open (≈1). The LSTM keeps it; the RNN washes it out through repeated `tanh` squashing.

In [ ]:
T = 60
# LSTM with forget gate held open and no new input: c_t = c_0
c = np.ones((1,1)); lstm_mem = []
for t in range(T):
    f = sigmoid(np.array([[3.0]]))     # forget gate ~0.95 (bias open)
    c = f*c + 0.0                       # no new input written
    lstm_mem.append(c.item())
# Vanilla RNN with small recurrent weight, signal decays
h = 1.0; rnn_mem = []
for t in range(T):
    h = np.tanh(0.9*h)
    rnn_mem.append(h)
plt.plot(lstm_mem, label='LSTM cell state (gate open)', color='#14b8a6')
plt.plot(rnn_mem, label='vanilla RNN hidden state', color='#f43f5e')
plt.xlabel('time step'); plt.ylabel('retained signal'); plt.legend()
plt.title('Long-term memory: LSTM cell vs vanilla RNN'); plt.show()

## Key takeaways

- The LSTM's additive cell update lets memory (and gradients) survive long gaps.
- Three gates (forget/input/output) learn what to keep, write, and expose.
- The GRU achieves the same with two gates and one state — fewer parameters.
- Opening the forget gate preserves a signal indefinitely, unlike a vanilla RNN.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — LSTM vs GRU parameter counts

Every gate is one dense layer over the concatenation $[\mathbf{x}; \mathbf{h}]$, costing $d_h (d_{in} + d_h) + d_h$ parameters. The LSTM has **4** gate-shaped blocks (forget, input, candidate, output); the GRU has **3** (reset, update, candidate). Implement both counts — the ratio is exactly $4/3$.

In [ ]:
def gate_params(d_in, d_h):
    """One gate: a dense layer on [x; h] -> d_h, with bias."""
    # TODO(you): d_h * (d_in + d_h) + d_h
    return ...


def lstm_params(d_in, d_h):
    # TODO(you): 4 gates
    return ...


def gru_params(d_in, d_h):
    # TODO(you): 3 gates
    return ...

In [ ]:
# Checks — run me
assert lstm_params(1, 1) == 12, "scalar LSTM: 4 gates x (1 + 1 weights + 1 bias)"
assert lstm_params(128, 256) == 4 * (256 * 384 + 256), "4 gates, each a dense layer on [x; h]"
assert gru_params(128, 256) == 3 * (256 * 384 + 256), "GRU drops one gate"
assert abs(lstm_params(128, 256) / gru_params(128, 256) - 4 / 3) < 1e-12, "exactly 4/3 the parameters"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def gate_params(d_in, d_h):
    return d_h * (d_in + d_h) + d_h


def lstm_params(d_in, d_h):
    return 4 * gate_params(d_in, d_h)


def gru_params(d_in, d_h):
    return 3 * gate_params(d_in, d_h)
```

</details>

### Exercise 2 — The cell-state highway

The LSTM's fix for vanishing gradients is **additive** memory:

$$c_t = f_t \, c_{t-1} + i_t \, \tilde{c}_t$$

Iterate the chain given per-step gate values. The checks demonstrate the whole argument: with $f = 1, i = 0$ the memory survives 100 steps untouched; with $f < 1$ the geometric decay of the vanilla RNN comes right back; and one open input gate writes the candidate into the cell.

In [ ]:
def cell_chain(c0, forgets, inputs, candidates):
    """Run the cell-state recurrence over aligned per-step gate values."""
    c = float(c0)

    for f, i, g in zip(forgets, inputs, candidates):
        # TODO(you): the additive update c = f*c + i*g
        c = ...

    return c

In [ ]:
# Checks — run me
T = 100
assert cell_chain(5.0, [1.0] * T, [0.0] * T, [0.0] * T) == 5.0, \
    "forget = 1, input = 0: the memory survives 100 steps untouched"
assert abs(cell_chain(5.0, [0.5] * 10, [0.0] * 10, [0.0] * 10) - 5.0 * 0.5 ** 10) < 1e-12, \
    "forget < 1 decays the memory geometrically — the vanilla RNN problem returns"
assert cell_chain(0.0, [1.0, 1.0, 1.0], [0.0, 1.0, 0.0], [0.0, 7.0, 0.0]) == 7.0, \
    "one open input gate writes the candidate into the cell"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def cell_chain(c0, forgets, inputs, candidates):
    c = float(c0)
    for f, i, g in zip(forgets, inputs, candidates):
        c = f * c + i * g
    return c
```

</details>